# 00 - Env + camera spike (T1, issue #2)

Recognition env spike: permission check, `source=0` stream, FPS measure.
No persistence: in-memory frames + score only (ADR-0002).

Pinned after spike (2026-09-24):
- ultralytics == 8.4.161
- torch == 2.14.0+cpu
- opencv-python == 5.0.0.93 (cv2 5.0.0)
- numpy == 2.4.6

Interpreter: `B:\venvs\baldur` (C: has <1GB free; torch/ultralytics do not fit on C:).
This machine has no camera: `source=0` does not open here, so the live path
emits the `error` event.

In [1]:
import json

In [2]:
class CameraError(Exception):
    """Camera failure carrying a machine-readable code."""

    def __init__(self, code, message):
        super().__init__(message)
        self.code = code
        self.message = message


def emit_error(code, message):
    """Print exactly one frozen `error` JSON line on stdout."""
    event = {"event": "error", "code": code, "message": message}
    print(json.dumps(event), flush=True)
    return event

In [3]:
def open_camera(source=0, open_capture=None):
    """Open the single webcam stream; raise CameraError when unavailable."""
    if open_capture is None:
        import cv2
        open_capture = cv2.VideoCapture
    capture = open_capture(source)
    if not capture.isOpened():
        capture.release()
        raise CameraError("CAMERA_UNAVAILABLE", f"source={source} could not be opened")
    return capture


def read_frame(capture):
    """Read one frame; raise CameraError when the stream fails."""
    ok, frame = capture.read()
    if not ok:
        raise CameraError("FRAME_READ_FAILED", "stream frame could not be read")
    return frame


def release_camera(capture):
    """Free the camera resource."""
    capture.release()

In [4]:
def measure_fps(frame_count, elapsed_seconds):
    """FPS from real timestamps; zero elapsed yields zero, never a crash."""
    if elapsed_seconds <= 0:
        return 0.0
    return frame_count / elapsed_seconds


def pinned_versions():
    """Installed runtime versions pinned by the env spike."""
    from importlib import metadata

    return {
        "ultralytics": metadata.version("ultralytics"),
        "torch": metadata.version("torch"),
        "opencv": metadata.version("opencv-python"),
        "numpy": metadata.version("numpy"),
    }

In [7]:
try:
    cap = open_camera()          # source=0
    frame = read_frame(cap)
    print(frame.shape)
    print("fps probe:", measure_fps(30, 2.0))
    print(pinned_versions())
    release_camera(cap)
except CameraError as e:
    emit_error(e.code, e.message)

(480, 640, 3)
fps probe: 15.0
{'ultralytics': '8.4.161', 'torch': '2.14.0+cpu', 'opencv': '5.0.0.93', 'numpy': '2.4.6'}
